In [1]:
import os
os.chdir("../")


In [2]:
%pwd

'/Users/swathivemuru/Documents/my_folder01/Rag/simple_chatbot'

In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

/opt/anaconda3/envs/medibot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Extracting text from PDF files
def  load_pdf_files(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [5]:
extracted_data=load_pdf_files("data")

In [8]:
extracted_data


[Document(metadata={'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': 'data/AI Engineering.pdf', 'total_pages': 535, 'page': 0, 'page_label': 'Cover'}, page_content='Chip Huyen\n AI Engineering\nBuilding Applications  \nwith Foundation Models'),
 Document(metadata={'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': 'data/AI Engineering.pdf', 'total_pages': 535, 'page': 1, 'page_label': 'Bac

In [9]:
len(extracted_data)

535

In [10]:
from typing import List
from langchain_core.documents import Document
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """given a list of documents, return a list of documents with only the page content and source metadata"""
    minimal_docs: List[Document] = []
    for doc in docs:
        src=doc.metadata.get("source")
        minimal_docs.append(Document(page_content=doc.page_content,metadata={"source": src}))      
        
    return minimal_docs


In [13]:
minimal_docs=filter_to_minimal_docs(extracted_data)

In [14]:
minimal_docs

[Document(metadata={'source': 'data/AI Engineering.pdf'}, page_content='Chip Huyen\n AI Engineering\nBuilding Applications  \nwith Foundation Models'),
 Document(metadata={'source': 'data/AI Engineering.pdf'}, page_content='9 781098 166304\n57999\nISBN:   978-1-098-16630-4\nUS $79.99    CAN $99.99\nDATA\nFoundation models have enabled many new AI use cases while lowering the barriers to entry for  \nbuilding AI products. This has transformed AI from an esoteric discipline into a powerful development \ntool that anyone can use—including those with no prior AI experience.\nIn this accessible guide, author Chip Huyen discusses AI engineering: the process of building applications \nwith readily available foundation models. AI application developers will discover how to navigate  \nthe AI landscape, including models, datasets, evaluation benchmarks, and the seemingly infinite number  \nof application patterns. The book also introduces a practical framework for developing an AI application \

In [16]:
# split the documents into smaller chunks 
def text_splitter(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    split_docs = text_splitter.split_documents(minimal_docs)
    return split_docs

In [17]:
split_docs=text_splitter(minimal_docs)
print(f"Number of documents after splitting: {len(split_docs)}")

Number of documents after splitting: 2602


In [18]:
split_docs

[Document(metadata={'source': 'data/AI Engineering.pdf'}, page_content='Chip Huyen\n AI Engineering\nBuilding Applications  \nwith Foundation Models'),
 Document(metadata={'source': 'data/AI Engineering.pdf'}, page_content='9 781098 166304\n57999\nISBN:   978-1-098-16630-4\nUS $79.99    CAN $99.99\nDATA\nFoundation models have enabled many new AI use cases while lowering the barriers to entry for  \nbuilding AI products. This has transformed AI from an esoteric discipline into a powerful development \ntool that anyone can use—including those with no prior AI experience.\nIn this accessible guide, author Chip Huyen discusses AI engineering: the process of building applications'),
 Document(metadata={'source': 'data/AI Engineering.pdf'}, page_content='with readily available foundation models. AI application developers will discover how to navigate  \nthe AI landscape, including models, datasets, evaluation benchmarks, and the seemingly infinite number  \nof application patterns. The book

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch
def download_embeddings():
    model_name = "BAAI/bge-small-en-v1.5"
    embeddings = HuggingFaceEmbeddings(model_name=model_name,
                                       model_kwargs={"device": "cuda"if torch.cuda.is_available() else "cpu"})
    return embeddings
embedding =download_embeddings()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8008.31it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
embedding

HuggingFaceEmbeddings(model_name='BAAI/bge-small-en-v1.5', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [21]:
vector=embedding.embed_query("Hello world")
vector

[0.015196045860648155,
 -0.02257068268954754,
 0.008547148667275906,
 -0.07417061924934387,
 0.003836351912468672,
 0.002713556634262204,
 -0.031267862766981125,
 0.044634055346250534,
 0.044055189937353134,
 -0.007871194742619991,
 -0.025200793519616127,
 -0.03336650878190994,
 0.014427892863750458,
 0.04653817042708397,
 0.008555099368095398,
 -0.01614585891366005,
 0.007405725307762623,
 -0.019012393429875374,
 -0.11472627520561218,
 -0.01815766841173172,
 0.12635934352874756,
 0.02970290556550026,
 0.025281090289354324,
 -0.03421781584620476,
 -0.040999747812747955,
 0.006617268547415733,
 0.010270575992763042,
 0.022362206131219864,
 0.004436279181391001,
 -0.12730960547924042,
 -0.016149243339896202,
 -0.020380215719342232,
 0.04721207544207573,
 0.011579904705286026,
 0.0681871771812439,
 0.007298700977116823,
 -0.01785298064351082,
 0.0407821349799633,
 -0.010269456543028355,
 0.02375710941851139,
 0.010602871887385845,
 -0.028584478422999382,
 0.008159652352333069,
 -0.0151804

In [22]:
print(len(vector))

384


In [23]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [25]:

from dotenv import load_dotenv
import os

load_dotenv(".env")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")



In [28]:
from pinecone import Pinecone



In [29]:
pinecone_api_key=PINECONE_API_KEY
pc=Pinecone(api_key=pinecone_api_key)
pc

In [30]:
from pinecone import ServerlessSpec
index_name="simple-chatbot"
if not pc.has_index(index_name):
    pc.create_index(name=index_name, dimension=384, metric="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index=pc.Index(index_name)

In [32]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(documents=split_docs, embedding=embedding, index_name=index_name)


In [ ]:
from langchain_pinecone import PineconeVectorStore
# load existing index
docsearch = PineconeVectorStore.from_existing_index(embedding=embedding, index_name=index_name)

In [ ]:
# Add more data to the existing index
# docsearch.add_documents(documents=[new_documents])
# this is how you can add more documents to the existing index. Just make sure that the new documents are in the same format as the existing ones and that they are properly embedded before adding them to the index.

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['PineconeVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x16c54d900>, search_kwargs={'k': 3})

In [ ]:
retrived_docs=retriever.invoke("what are embeddings?")
retrived_docs

[Document(id='91d5fa70-c9d4-4f25-bcd1-f8ea66a36647', metadata={'source': 'data/AI Engineering.pdf'}, page_content='rerank all retrieved candidates, and caches to reduce latency.8\nWith embedding-based retrieval, we again encounter embeddings, which are dis‐\ncussed in Chapter 3. As a reminder, an embedding is typically a vector that aims to\npreserve the important properties of the original data. An embedding-based retriever\ndoesn’t work if the embedding model is bad.\nRAG | 261'),
 Document(id='336607ec-4c9d-447a-b0e9-6eb5e595baef', metadata={'source': 'data/AI Engineering.pdf'}, page_content='rerank all retrieved candidates, and caches to reduce latency.8\nWith embedding-based retrieval, we again encounter embeddings, which are dis‐\ncussed in Chapter 3. As a reminder, an embedding is typically a vector that aims to\npreserve the important properties of the original data. An embedding-based retriever\ndoesn’t work if the embedding model is bad.\nRAG | 261'),
 Document(id='f521c00f-d

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="llama-3.3-70b-versatile"
)

In [ ]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [ ]:
system_prompt = """
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, say that you don't know.
Use three sentences maximum and keep the answer concise.

{context}
"""

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [ ]:
response=rag_chain.invoke("what is rag?")
print(response.content)

RAG is a technique that enhances a model's generation by retrieving relevant information from external memory sources. It uses external memory to improve the model's performance. RAG stands for Retrieve, Augment, Generate, which describes its functionality.
